# Hann Qian — pitch correction starter

Hi Hann Qian! This is your starting point, not a finished model. We load the handover tables, check labels, and measure what happens when we change nothing. Your model choice and training are still yours to do.

**Labels are PROVISIONAL pending group manual review.** Please review a few examples with the group before treating the targets as ground truth. This notebook does not fit a model and never opens test data.

### Keep the boundary clear
- `X` contains only prediction-derived allowlisted features. Reference matches, labels, IDs and split metadata are not model inputs.
- Note features: `start, end, pitch, amplitude, duration, prev_gap, next_gap`; pair features: `gap, pitch_delta, duration_sum`.
- Missing/ambiguous labels are excluded only from this task's training-label diagnostics, never from full end-to-end evaluation.
- Timing targets are absolute seconds; pitch targets are absolute MIDI. Positive classifier labels mean merge the pair / remove the false note.


In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

# Open from this notebook folder, or run from the project root.
HERE = next(p / 'notebooks' / 'correction_handover' for p in [Path.cwd(), *Path.cwd().parents, Path.cwd() / 'GTSinger_100_unique_sample'] if (p / 'selected_100.csv').exists())
sys.path.insert(0, str(HERE))
import correction_utils as cu
DATA = HERE / '..' / '..' / 'basic_pitch_runs' / 'correction_handover'
notes = pd.read_csv(DATA / 'train_features.csv', dtype={'note_id':str,'recording_id':str,'sample_id':str})
note_targets = pd.read_csv(DATA / 'proposed_v2' / 'train_targets_v2.csv', dtype={'note_id':str}).rename(columns={'false_note':'false_label','reference_start':'target_start','reference_end':'target_end','reference_pitch':'target_pitch'})
assert notes['split'].eq('train').all(), 'Only train data belongs in this starter.'
manifest = pd.read_csv(DATA / 'training_recording_manifest.csv')
cu.validate_notes(notes, manifest)
features, targets = notes, note_targets
TASK = 'pitch'
X, joined = cu.training_data(features, targets, TASK)
print('Feature allowlist:', X.columns.tolist())
print('Rows:', len(X), '| target-row coverage: 100%')
display(X.head())

Feature allowlist: ['start', 'end', 'pitch', 'amplitude', 'duration', 'prev_gap', 'next_gap']
Rows: 7623 | target-row coverage: 100%


,start,end,pitch,amplitude,duration,prev_gap,next_gap
0,0.394739,0.684989,55.0,0.729802,0.290249,0.00000,0.092880
1,0.777868,0.986848,49.0,0.616202,0.208980,0.09288,0.034830
2,1.021678,1.265488,54.0,0.627965,0.243810,0.03483,0.150930
3,1.416417,1.625397,49.0,0.576493,0.208980,0.15093,0.092880
4,1.718277,1.915646,55.0,0.518624,0.197370,0.09288,0.082554


## Check what we can actually learn from

Target rows must cover every feature ID, even when their task labels are missing. For classification, inspect both class counts before choosing a method. For timing/pitch, the value counts below describe target coverage rather than classifier classes.

In [2]:
label_columns = cu.TARGETS[TASK]
usable = joined[label_columns].notna().all(axis=1)
if 'ambiguous' in joined:
    # CSV accepts True/False or 1/0. Reject unknown encodings rather than guessing.
    ambiguity = joined['ambiguous'].astype(str).str.lower().map({'true':True,'false':False,'1':True,'0':False})
    assert ambiguity.notna().all(), 'ambiguous must be explicit True/False or 1/0'
    usable &= ~ambiguity
print({'total_rows':len(joined), 'usable_task_labels':int(usable.sum()),
       'excluded_task_labels':int((~usable).sum())})
for column in label_columns:
    print(column, 'non-null coverage:', float(joined[column].notna().mean()))
    display(joined.loc[usable,column].value_counts(dropna=False).rename('count'))
labelled = joined.loc[usable].copy()
X_task = X.loc[usable].copy()  # Only these allowlisted columns may reach your future model.
folds = pd.read_csv(DATA / 'folds.csv').set_index('sample_id')['fold']
groups = labelled['sample_id'].copy()
fold_ids = groups.map(folds)
assert fold_ids.notna().all(), 'Missing training-song fold'
print('Labelled examples per song-grouped fold:', fold_ids.value_counts().sort_index().to_dict())
print('Do not fit until the group approves the provisional label version.')


{'total_rows': 7623, 'usable_task_labels': 3697, 'excluded_task_labels': 3926}
target_pitch non-null coverage: 0.4849796667978486


target_pitch
61.0    290
63.0    288
65.0    284
59.0    277
62.0    258
64.0    245
66.0    210
68.0    197
60.0    192
58.0    174
67.0    174
57.0    155
56.0    153
55.0    152
54.0    109
69.0    107
70.0     83
52.0     69
53.0     55
50.0     47
51.0     37
49.0     30
71.0     29
72.0     25
73.0     21
48.0     11
75.0      5
74.0      5
47.0      5
44.0      3
46.0      3
45.0      2
77.0      1
43.0      1
Name: count, dtype: int64

Labelled examples per song-grouped fold: {0: 678, 1: 856, 2: 827, 3: 671, 4: 665}
Do not fit until the group approves the provisional label version.


## Our first comparison: do nothing

This is a real no-change baseline on usable training task labels, not evidence of validation performance. Empty-label metrics have no useful interpretation. Nothing is fitted here.

In [3]:
if len(labelled) == 0:
    baseline = {'status':'BLOCKED_NO_REVIEWED_OR_PROVISIONAL_TARGETS', 'n':0, 'metrics':None}
else:
    baseline = cu.pitch_metrics(labelled['target_pitch'], labelled['pitch'])
print('No-change task baseline (not a trained model):', baseline)


No-change task baseline (not a trained model): {'n': 3697, 'mae_cents': 8.574519880984582, 'mae_semitones': 0.08574519880984582, 'octave_error_rate': 0.0, 'rmse_cents': 31.592824705887626, 'within_50_cents': 0.9210170408439275}


## Your model goes here — later

Use `X_task` and the separate labels when you are ready. Keep validation separate by song (`sample_id`), choose settings on train/val only, and leave test untouched. Do not infer features from reference alignment. For merge predictions, resolve overlapping pair decisions before producing a complete note table.

## Pitch model (Hann Qian)

**Model:** XGBoost classifier with 3 classes per note: shift down 1 semitone / keep / shift up 1 semitone. A note is changed only when the model is confident enough (cutoff chosen on train CV). Labels: `proposed_v2/train_targets_v2.csv`.

**Features:**
- the 7 allowlisted note features
- context: interval to the previous/next predicted note, pitch relative to the recording's median
- pyin (whole note): how many cents the *sung* pitch sits above/below the predicted MIDI note, measured from the audio with `librosa.pyin` (`hann_qian_pitch_model/pyin_features.py`)

Context and pyin go **beyond the handover allowlist**. They come from predictions and audio only, never from references or labels; flagged to the group.

**Why:** most pitch errors are exactly ±1 semitone. The allowlisted features can spot *which* notes look wrong but not sharp vs flat; the pyin offset gives the direction (right sign on 87% of wrong notes).

**Honest expectation:** nested CV (cutoff chosen inside the training folds) gave about **+35 net notes fixed of 292 pitch errors (~12%)**, pitch accuracy 92.1% → 93.1%. The cutoff table below picks its cutoff on the same CV predictions it reports, so it is slightly optimistic. Trimming notes to their middle was tested and did not help (`hann_qian_pitch_model/results_middle.txt`).

**Val:** the handover has no val references, so val corrections are exported for the shared evaluator, not scored here. Test is not touched.

In [4]:
from xgboost import XGBClassifier

MODEL_DIR = HERE / 'hann_qian_pitch_model'
CONTEXT = ['prev_int', 'next_int', 'rec_med']
PYIN = ['f0_voiced_frac', 'f0_cents_med', 'f0_cents_iqr', 'f0_frac_up', 'f0_frac_down', 'f0_has']
FEATURES = cu.NOTE_FEATURES + CONTEXT + PYIN


def add_features(frame, split):
    """Allowlisted features + context + whole-note pyin. Predictions and audio only; never references/labels."""
    pyin = pd.read_csv(MODEL_DIR / f'{split}_pyin_features.csv', dtype={'note_id':str})
    assert not pyin.note_id.duplicated().any() and set(pyin.note_id) == set(frame.note_id)
    out = frame.join(pyin.set_index('note_id'), on='note_id').sort_values(['recording_id', 'start'])
    g = out.groupby('recording_id')
    out['prev_int'] = out.pitch - g.pitch.shift(1)          # interval from previous predicted note
    out['next_int'] = g.pitch.shift(-1) - out.pitch         # interval to next predicted note
    out['rec_med'] = out.pitch - g.pitch.transform('median')  # pitch relative to the recording
    out[CONTEXT + PYIN] = out[CONTEXT + PYIN].fillna(0)
    return out.loc[frame.index]                              # back to the original row order


train_X = add_features(notes, 'train')[FEATURES].astype(float)
assert np.allclose(train_X[cu.NOTE_FEATURES].to_numpy(), X.to_numpy())  # allowlisted part unchanged
print('Features:', FEATURES)
display(train_X.head())

Features: ['start', 'end', 'pitch', 'amplitude', 'duration', 'prev_gap', 'next_gap', 'prev_int', 'next_int', 'rec_med', 'f0_voiced_frac', 'f0_cents_med', 'f0_cents_iqr', 'f0_frac_up', 'f0_frac_down', 'f0_has']


,start,end,pitch,amplitude,duration,prev_gap,next_gap,prev_int,next_int,rec_med,f0_voiced_frac,f0_cents_med,f0_cents_iqr,f0_frac_up,f0_frac_down,f0_has
0,0.394739,0.684989,55.0,0.729802,0.290249,0.00000,0.092880,0.0,-6.0,1.0,1.00,0.0,10.0,0.040000,0.000000,1.0
1,0.777868,0.986848,49.0,0.616202,0.208980,0.09288,0.034830,-6.0,5.0,-5.0,1.00,10.0,35.0,0.157895,0.157895,1.0
2,1.021678,1.265488,54.0,0.627965,0.243810,0.03483,0.150930,5.0,-5.0,0.0,0.95,30.0,30.0,0.052632,0.000000,1.0
3,1.416417,1.625397,49.0,0.576493,0.208980,0.15093,0.092880,-5.0,6.0,-5.0,1.00,35.0,65.0,0.333333,0.000000,1.0
4,1.718277,1.915646,55.0,0.518624,0.197370,0.09288,0.082554,6.0,-6.0,1.0,1.00,-50.0,20.0,0.000000,0.411765,1.0


In [5]:
# Song-grouped 5-fold CV on train: every note gets a prediction from a model that never saw its song.
MODEL_PARAMS = dict(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8,
                    colsample_bytree=0.8, random_state=0, n_jobs=4)
CUTOFFS = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]

delta = joined['target_pitch'] - joined['pitch']   # true shift in semitones (NaN = no label)
y = delta.clip(-1, 1) + 1                          # classes 0, 1, 2 = shift -1, keep, +1
note_folds = notes['sample_id'].map(folds)
oof = np.zeros((len(notes), 3))
for k in sorted(note_folds.unique()):
    tr, te = usable & (note_folds != k), note_folds == k
    oof[te.to_numpy()] = XGBClassifier(**MODEL_PARAMS).fit(train_X[tr], y[tr].astype(int)).predict_proba(train_X[te])


def to_shift(P, cutoff):
    """Shift by the predicted class only when the model is at least `cutoff` confident; otherwise keep."""
    cls = P.argmax(1) - 1
    return np.where((cls != 0) & (P.max(1) >= cutoff), cls, 0)


def fixed_broken(shift, rows):
    d, t = shift[rows], delta.to_numpy()[rows]
    return int(((d != 0) & (d == t)).sum()), int(((d != 0) & (d != t)).sum())


rows = usable.to_numpy()
cutoff_table = pd.DataFrame([dict(cutoff=c, fixed=f, broken=b, net=f - b)
                             for c in CUTOFFS for f, b in [fixed_broken(to_shift(oof, c), rows)]])
display(cutoff_table)
CUTOFF = float(cutoff_table.loc[cutoff_table.net.idxmax(), 'cutoff'])
print('Chosen cutoff (train CV only):', CUTOFF)

,cutoff,fixed,broken,net
0,0.50,76,47,29
1,0.60,68,28,40
2,0.70,55,15,40
3,0.80,36,9,27
4,0.90,19,4,15
5,0.95,6,1,5


Chosen cutoff (train CV only): 0.6


In [6]:
# Cross-validated result on train (song-grouped folds), using the chosen cutoff.
cv_shift = to_shift(oof, CUTOFF)
cv_corrected = notes[cu.NOTE_COLUMNS].copy(deep=True)
cv_corrected['pitch'] = cv_corrected['pitch'] + cv_shift
fixed, broken = fixed_broken(cv_shift, rows)
print(f'Wrong-pitch labelled notes: {int((delta[usable] != 0).sum())} of {int(usable.sum())}')
print(f'Fixed {fixed}, broken {broken}, net {fixed - broken}')
print('No change :', cu.pitch_metrics(labelled['target_pitch'], labelled['pitch']))
print('CV model  :', cu.pitch_metrics(labelled['target_pitch'], cv_corrected.loc[usable, 'pitch']))

full_manifest = pd.read_csv(DATA / 'training_recording_manifest.csv')
full_references = pd.read_csv(DATA / 'references_train.csv')
print('End-to-end, no change:', cu.evaluate_end_to_end(notes[cu.NOTE_COLUMNS], full_references, full_manifest)['summary'])
print('End-to-end, CV model :', cu.evaluate_end_to_end(cv_corrected, full_references, full_manifest)['summary'])

Wrong-pitch labelled notes: 292 of 3697
Fixed 68, broken 28, net 40
No change : {'n': 3697, 'mae_cents': 8.574519880984582, 'mae_semitones': 0.08574519880984582, 'octave_error_rate': 0.0, 'rmse_cents': 31.592824705887626, 'within_50_cents': 0.9210170408439275}
CV model  : {'n': 3697, 'mae_cents': 7.438463619150663, 'mae_semitones': 0.07438463619150662, 'octave_error_rate': 0.0, 'rmse_cents': 29.831371715638056, 'within_50_cents': 0.9323776034622667}


End-to-end, no change: {'song_macro_f1': 0.26275846642410683, 'song_macro_offset_f1': 0.15386648702425776, 'song_macro_precision': 0.27330092213557544, 'song_macro_recall': 0.2582440522033118, 'micro_f1': 0.26521100683138604, 'songs': 60, 'recordings': 534}


End-to-end, CV model : {'song_macro_f1': 0.2651682854841605, 'song_macro_offset_f1': 0.15531743035251114, 'song_macro_precision': 0.2758240111128115, 'song_macro_recall': 0.26061233387257193, 'micro_f1': 0.2677647960160889, 'songs': 60, 'recordings': 534}


In [7]:
# Final model: train on ALL labelled train notes, apply once to val, export complete note tables.
final_model = XGBClassifier(**MODEL_PARAMS).fit(train_X[usable], y[usable].astype(int))

val_notes = pd.read_csv(DATA / 'val_features.csv', dtype={'note_id':str,'recording_id':str,'sample_id':str})
assert val_notes['split'].eq('val').all()
all_recordings = pd.read_csv(DATA / 'recording_manifest.csv', dtype={'recording_id':str,'sample_id':str})
val_manifest = all_recordings[all_recordings.split.eq('val')]
cu.validate_notes(val_notes, val_manifest)
val_X = add_features(val_notes, 'val')[FEATURES]
val_shift = to_shift(final_model.predict_proba(val_X), CUTOFF)
val_corrected = val_notes[cu.NOTE_COLUMNS].copy(deep=True)
val_corrected['pitch'] = val_corrected['pitch'] + val_shift
print('Val notes:', len(val_notes), '| pitch changed:', int((val_shift != 0).sum()),
      '| up:', int((val_shift > 0).sum()), '| down:', int((val_shift < 0).sum()))

# Train corrections use the cross-validated (out-of-fold) model, so no song is corrected by a model trained on it.
for name, corrected, manifest_ in [('hann_qian_pitch_val_corrected.csv', val_corrected, val_manifest),
                                    ('hann_qian_pitch_train_corrected_cv.csv', cv_corrected, full_manifest)]:
    path = HERE / 'exports' / name
    if path.exists():
        print('Exists, not overwritten (delete it to re-export):', path)
    else:
        cu.export_corrections(corrected, manifest_, path)
        print('Exported:', path)

Val notes: 2329 | pitch changed: 83 | up: 29 | down: 54
Exported: C:\Users\qhanweil\Downloads\GTSinger_Correction_Group_Handover\GTSinger_100_unique_sample\notebooks\correction_handover\exports\hann_qian_pitch_val_corrected.csv
Exported: C:\Users\qhanweil\Downloads\GTSinger_Correction_Group_Handover\GTSinger_100_unique_sample\notebooks\correction_handover\exports\hann_qian_pitch_train_corrected_cv.csv


## Try the shared output interface

The CSV below contains the real original notes unchanged. A future correction CSV is the **complete replacement note set**, not a patch: omitted notes are deletions; for a merge retain one ID and omit absorbed IDs. Never overwrite the source tables.

In [8]:
# Real, unchanged notes: do not filter this with the task-label mask.
unchanged = notes[cu.NOTE_COLUMNS].copy(deep=True)
output = HERE / 'exports' / 'hann_qian_pitch_unchanged.csv'
if output.exists():
    # Safe notebook rerun: verify the existing export instead of overwriting it.
    existing = pd.read_csv(output, dtype={'note_id':str,'recording_id':str})
    pd.testing.assert_frame_equal(cu.validate_notes(existing, manifest), cu.validate_notes(unchanged, manifest), check_dtype=False)
else:
    cu.export_corrections(unchanged, manifest, output)
print('Unchanged-note interface example:', output)


Unchanged-note interface example: C:\Users\qhanweil\Downloads\GTSinger_Correction_Group_Handover\GTSinger_100_unique_sample\notebooks\correction_handover\exports\hann_qian_pitch_unchanged.csv


## Full end-to-end scoring (when the evaluator tables are ready)

Use `cu.evaluate_end_to_end(corrected, full_references, evaluation_manifest)` for both the unchanged baseline and future corrected notes. All three tables must describe the same declared train/val scope. `evaluation_manifest` has `recording_id, sample_id, split` and must include recordings with zero predictions. `full_references` has `note_id, recording_id, start, end, pitch`; reference IDs are independent of prediction IDs.

Do **not** build this manifest from a filtered prediction table, or filter references with `usable`, task matches, or ambiguity flags. The manifest loaded above is the complete training recording manifest, including zero-prediction recordings. Reference tables are evaluator-only, never feature inputs.

The shared evaluator reuses `bp_tuning_core.score_notes` and `aggregate`: 50 ms onset, 50-cent pitch, optional offsets (20% or 50 ms), song-pooled macro scores and pooled micro F1. The next cell runs it on every training recording using evaluator-only references. It never filters on task-label availability. No validation or test labels are opened.


In [9]:
full_manifest = pd.read_csv(DATA / 'training_recording_manifest.csv')
full_references = pd.read_csv(DATA / 'references_train.csv')
end_to_end = cu.evaluate_end_to_end(unchanged, full_references, full_manifest)
print('All-training no-correction baseline:', end_to_end['summary'])
assert end_to_end['summary']['recordings'] == len(full_manifest)


All-training no-correction baseline: {'song_macro_f1': 0.26275846642410683, 'song_macro_offset_f1': 0.15386648702425776, 'song_macro_precision': 0.27330092213557544, 'song_macro_recall': 0.2582440522033118, 'micro_f1': 0.26521100683138604, 'songs': 60, 'recordings': 534}
